# 1. Load Data

In [20]:
import pandas as pd # Pandas is used for data manipulation and analysis

orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')

# create revenue

items['revenue'] = items['price']

df = items.merge(orders, on='order_id')
df = df.merge(products, on='product_id')

# 2. Data Overview

In [21]:
df.shape # Show quantity of lines and columns

(112650, 23)

In [22]:
df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,revenue,customer_id,order_status,...,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,58.90,3ce436f183e68e07877b285a838db11a,delivered,...,2017-09-20 23:43:48,2017-09-29 00:00:00,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,239.90,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,2017-05-12 16:04:24,2017-05-15 00:00:00,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,199.00,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,2018-01-22 13:19:16,2018-02-05 00:00:00,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,12.99,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,2018-08-14 13:32:39,2018-08-20 00:00:00,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,199.90,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,2017-03-01 16:42:31,2017-03-17 00:00:00,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       112650 non-null  str    
 1   order_item_id                  112650 non-null  int64  
 2   product_id                     112650 non-null  str    
 3   seller_id                      112650 non-null  str    
 4   shipping_limit_date            112650 non-null  str    
 5   price                          112650 non-null  float64
 6   freight_value                  112650 non-null  float64
 7   revenue                        112650 non-null  float64
 8   customer_id                    112650 non-null  str    
 9   order_status                   112650 non-null  str    
 10  order_purchase_timestamp       112650 non-null  str    
 11  order_approved_at              112635 non-null  str    
 12  order_delivered_carrier_date   111456 non

# 3. Top products Analysis

In [43]:
top_products = df.groupby('product_category_name')['revenue'].sum().sort_values(ascending=False).head(10)
# Identifying the top 10 products that generated the most revenue
top_products

product_category_name
beleza_saude              1258681.34
relogios_presentes        1205005.68
cama_mesa_banho           1036988.68
esporte_lazer              988048.97
informatica_acessorios     911954.32
moveis_decoracao           729762.49
cool_stuff                 635290.85
utilidades_domesticas      632248.66
automotivo                 592720.11
ferramentas_jardim         485256.46
Name: revenue, dtype: float64

# 4. Time-based analysis

In [42]:
# Identify categories with strong and consistent monthly revenue

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['month'] = df['order_purchase_timestamp'].dt.to_period('M')

monthly = df.groupby(['product_category_name', 'month'])['revenue'].sum()

median = monthly.groupby('product_category_name').median()
std = monthly.groupby('product_category_name').std()
rv = std / median

result = pd.DataFrame({
    'median_revenue': median,
    'std_revenue': std,
    'relative_variation': rv
})

result['score'] = result['median_revenue'] / result['relative_variation']

top = result.sort_values(by='score', ascending=False).head(10)

top

,median_revenue,std_revenue,relative_variation,score
product_category_name,,,,
cama_mesa_banho,55082.820,24522.585956,0.445195,123727.451279
relogios_presentes,63462.180,34532.344732,0.544141,116628.289265
esporte_lazer,49751.450,23446.606808,0.471275,105567.803364
cool_stuff,33696.470,13382.996239,0.397163,84842.890948
beleza_saude,50705.775,36068.305092,0.711325,71283.516423
automotivo,31370.690,15052.907281,0.479840,65377.416649
informatica_acessorios,40293.930,25785.625869,0.639938,62965.343681
moveis_decoracao,31425.220,17106.044032,0.544341,57730.732493
ferramentas_jardim,23747.150,9779.049711,0.411799,57666.864344
